In [10]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

In [3]:
%pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   -- ------------------------------------- 2.6/48.9 MB 12.2 MB/s eta 0:00:04
   --- ------------------------------------ 3.9/48.9 MB 12.3 MB/s eta 0:00:04
   ---- ----------------------------------- 6.0/48.9 MB 9.3 MB/s eta 0:00:05
   ------ --------------------------------- 8.4/48.9 MB 9.9 MB/s eta 0:00:05
   -------- ------------------------------- 11.0/48.9 MB 10.5 MB/s eta 0:00:04
   ----------- ---------------------------- 13.9/48.9 MB 11.1 MB/s eta 0:00:04
   ------------- -------------------------- 16.8/48.9 MB 11.4 MB/s eta 0:00:03
   ---------------- ----------------------- 19.7/48.9 MB 11.6 MB/s eta 0:00:03
   ------------------ --------------------- 22.5/48.9 MB 11.9 MB/s eta 0:00:03
   --------------------- ------------------ 26.0/48.9 MB 12.2 MB/s eta 0:00:02
   ----------------------- ---------------- 29.1/48.9 MB 12.5 MB/s eta 0:00:


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
df = pd.read_csv("churnData_Fixed.csv")

In [15]:
print("Loaded shape:", df.shape)

Loaded shape: (7043, 36)


In [16]:
def get_rolling_splits(data, window_col="window_id"):
    windows = sorted(data[window_col].unique())
    splits = []
    for i in range(len(windows) - 1):
        train = data[data[window_col] == windows[i]].copy()
        test = data[data[window_col] == windows[i + 1]].copy()
        splits.append((train, test))
    return splits
 
splits = get_rolling_splits(df)
print(f"Built {len(splits)} rolling train/test window pairs")

Built 9 rolling train/test window pairs


In [17]:
leak_cols = ["signup_date", "year_month", "window_id"]
model_features = [c for c in df.columns if c not in leak_cols + ["Churn"]]
 
print(f"\nUsing {len(model_features)} features")


Using 32 features


In [18]:
train_window, test_window = splits[0]
 
X_train_full = train_window[model_features]
y_train_full = train_window["Churn"]
 
X_test = test_window[model_features]
y_test = test_window["Churn"]
 
print(f"\nTraining on window 0 ({len(X_train_full)} customers)")
print(f"Testing on window 1 ({len(X_test)} customers)")


Training on window 0 (705 customers)
Testing on window 1 (704 customers)


In [19]:
X_train, X_calib, y_train, y_calib = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2, random_state=42, stratify=y_train_full
)
 
print(f"\nSplit window 0 into: {len(X_train)} for training, {len(X_calib)} for calibration")


Split window 0 into: 564 for training, 141 for calibration


In [20]:
base_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    eval_metric="logloss",
    random_state=42,
)
base_model.fit(X_train, y_train)
 
print("\nBase XGBoost model trained.")
 


Base XGBoost model trained.


In [21]:
y_pred = base_model.predict(X_test)
y_proba_raw = base_model.predict_proba(X_test)[:, 1]  # raw confidence, not yet calibrated
 
print("\n--- Baseline performance (window 0 -> window 1), UNCALIBRATED ---")
print("Accuracy: ", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:   ", round(recall_score(y_test, y_pred), 4))
print("F1:       ", round(f1_score(y_test, y_pred), 4))
print("AUC-ROC:  ", round(roc_auc_score(y_test, y_proba_raw), 4))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nFull report:\n", classification_report(y_test, y_pred))


--- Baseline performance (window 0 -> window 1), UNCALIBRATED ---
Accuracy:  0.8665
Precision: 0.2264
Recall:    0.1846
F1:        0.2034
AUC-ROC:   0.746

Confusion matrix:
 [[598  41]
 [ 53  12]]

Full report:
               precision    recall  f1-score   support

           0       0.92      0.94      0.93       639
           1       0.23      0.18      0.20        65

    accuracy                           0.87       704
   macro avg       0.57      0.56      0.57       704
weighted avg       0.85      0.87      0.86       704



In [22]:
try:
    # scikit-learn 1.6+: wrap the fitted model in FrozenEstimator
    from sklearn.frozen import FrozenEstimator
    calibrated_model = CalibratedClassifierCV(FrozenEstimator(base_model), method="sigmoid")
except ImportError:
    # older scikit-learn: cv="prefit" does the same thing
    calibrated_model = CalibratedClassifierCV(base_model, method="sigmoid", cv="prefit")
 
calibrated_model.fit(X_calib, y_calib)
 
print("\nPlatt scaling calibration fitted.")


Platt scaling calibration fitted.


In [23]:
y_proba_calibrated = calibrated_model.predict_proba(X_test)[:, 1]
y_pred_calibrated = (y_proba_calibrated >= 0.5).astype(int)
 
print("\n--- Calibrated performance (window 0 -> window 1) ---")
print("Accuracy: ", round(accuracy_score(y_test, y_pred_calibrated), 4))
print("Precision:", round(precision_score(y_test, y_pred_calibrated), 4))
print("Recall:   ", round(recall_score(y_test, y_pred_calibrated), 4))
print("F1:       ", round(f1_score(y_test, y_pred_calibrated), 4))
print("AUC-ROC:  ", round(roc_auc_score(y_test, y_proba_calibrated), 4))


--- Calibrated performance (window 0 -> window 1) ---
Accuracy:  0.8324
Precision: 0.2427
Recall:    0.3846
F1:        0.2976
AUC-ROC:   0.746


In [24]:
calib_check = pd.DataFrame({
    "predicted_proba": y_proba_calibrated,
    "actual_churn": y_test.values
})
calib_check["bucket"] = pd.cut(calib_check["predicted_proba"], bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0])
 
print("\n--- Calibration sanity check (predicted bucket vs. actual churn rate) ---")
print(calib_check.groupby("bucket").agg(
    avg_predicted=("predicted_proba", "mean"),
    actual_churn_rate=("actual_churn", "mean"),
    count=("actual_churn", "size")
))


--- Calibration sanity check (predicted bucket vs. actual churn rate) ---
            avg_predicted  actual_churn_rate  count
bucket                                             
(0.0, 0.2]       0.042458           0.061453    537
(0.2, 0.4]       0.288966           0.120000     50
(0.4, 0.6]       0.508421           0.133333     30
(0.6, 0.8]       0.701445           0.304348     23
(0.8, 1.0]       0.935809           0.234375     64


C:\Users\Bhushan\AppData\Local\Temp\ipykernel_18808\2986496698.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(calib_check.groupby("bucket").agg(
